# Primera entrega piloto: feature engineering con ranking + LightGBM (BO, 20 semillas)

**Flujo**
1. Carga de datos y chequeos
2. Feature engineering: `ccajas_depositos` roto en 202105, lags y deltas 1 y 2, y ranking con cero fijo dentro de cada mes
3. Diagnóstico de data drift (sobre las variables rankeadas)
4. Optimización bayesiana: entrena con **marzo + abril**, valida en **junio** (métrica AUC, una sola semilla)
5. Curva de ganancia de junio con `n_semillas` semillas, para elegir el corte
6. Modelo final con **marzo + abril + mayo + junio** y `n_semillas` semillas; la probabilidad de agosto es el promedio
7. CSV de envío: uno por defecto (corte de la curva suavizada) y uno por cada valor de `PARAM$cortes_envios`

**Cambios respecto del notebook base**
* Feature engineering antes de cualquier entrenamiento; el modelo usa las variables rankeadas más `numero_de_cliente` y `foto_mes` sin transformar (sin ranking ni lags).
* `cprestamos_personales` y `mprestamos_personales` pasan por el feature engineering, pero **no se usan en la BO**; sí en la curva de junio y en el modelo final.
* El dataset completo con el feature engineering se guarda en CSV (`PARAM$archivo_dataset_fe`) para poder volver a cargarlo sin repetir el FE.
* Sin undersampling en ningún paso (BO, curva y modelo final).
* `feature_fraction_bynode = 0.20` fijo; la BO optimiza `num_iterations`, `num_leaves` y `min_sum_hessian_in_leaf`.
* Métrica de la BO: AUC. La BO corre con una sola semilla.
* `PARAM$semillas_por_primigenia` define las semillas de la curva de junio y del modelo final (las mismas en ambos): 5 generadas con la primigenia 501367, 10 con 898543 y 5 con 500153, 20 en total (`PARAM$n_semillas`). La BO usa una sola semilla (`PARAM$semilla_primigenia`, 898543).
* La BO hace 10 evaluaciones.

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Oct 10 17:09:03 2026"

## 1. Inicio y librerías

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE))
gc(full=TRUE, verbose=FALSE)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671205,35.9,1479650,79.1,1479650,79.1
Vcells,1242606,9.5,8388608,64.0,1978693,15.1


In [3]:
# Si falta algun paquete, descomentar:
#install.packages(c("data.table", "bbotk", "mlr3", "mlr3mbo", "mlr3learners", "DiceKriging", "lightgbm"))

library(data.table)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
library(paradox)
library(DiceKriging)
library(lightgbm)


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: paradox

Loading required package: mlr3tuning


Attaching package: ‘DiceKriging’


The following object is masked from ‘package:bbotk’:

    branin




## 2. Parámetros

Todo lo que se puede querer cambiar está en esta celda.

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 898543


# semillas para la curva de junio y para el modelo final (la BO usa una sola semilla: semilla_primigenia)
# nombre = semilla primigenia, valor = cantidad de semillas que se generan a partir de ella
PARAM$semillas_por_primigenia <- c("501367"= 5, "898543"= 10, "500153"= 5)
PARAM$n_semillas <- sum(PARAM$semillas_por_primigenia)   # 20 en total
PARAM$experimento <- paste0("primer_entrega_piloto_FE_rank_", PARAM$n_semillas, "semillas")

# las semillas se generan de forma reproducible: para cada primigenia, set.seed() + sample()
PARAM$semillas <- unlist(lapply(names(PARAM$semillas_por_primigenia), function(prim) {
  set.seed(as.integer(prim))
  sample(100000:999999, PARAM$semillas_por_primigenia[[prim]])
}), use.names= FALSE)

# la curva de junio y el modelo final usan las mismas semillas
stopifnot(PARAM$n_semillas >= 1, length(PARAM$semillas) == PARAM$n_semillas,
          !anyDuplicated(PARAM$semillas))

# --- RUTAS ADAPTADAS PARA VM DE CLOUD ---
PARAM$carpeta_base <- path.expand("~/buckets/b1")
PARAM$archivo_dataset <- file.path(PARAM$carpeta_base, "datasets/competencia_01_crudo.csv")

# dataset completo con el feature engineering (mismo lugar que el original, con FE en el nombre)
PARAM$archivo_dataset_fe <- file.path(dirname(PARAM$archivo_dataset), "competencia_01_crudo_FE_rank.csv")

# meses
PARAM$meses_fe    <- 202103:202108                       # meses con los que se hace el feature engineering
PARAM$train_bo    <- c(202103, 202104)                   # entrena la BO (sin mayo a proposito)
PARAM$validate_bo <- c(202106)                           # valida la BO y arma la curva de ganancia
PARAM$train_final <- c(202103, 202104, 202105, 202106)   # modelo final
PARAM$future      <- c(202108)                           # mes a predecir (sin clase)

# variables que pasan por el feature engineering pero NO se usan en la Bayesian Optimization
# (si en la curva de junio y el modelo final); numero_de_cliente y foto_mes no tienen FE y se usan en todo el proceso
PARAM$variables_fuera_bo <- c("cprestamos_personales", "mprestamos_personales")

# columnas ya rankeadas que se quieren sacar del modelo (nombres exactos, por ejemplo "mcuentas_saldo_lag1_rank")
PARAM$columnas_excluidas_modelo <- character(0)

# cantidad de envios permitida por la campania
PARAM$envios_min <- 8000
PARAM$envios_max <- 15000
PARAM$envios_manual <- NA    # poner un numero (8000-15000) para reemplazar el corte por defecto (el de la curva suavizada)

# ademas del CSV por defecto, se escribe un CSV por cada uno de estos cortes
PARAM$cortes_envios <- seq(8000, 15000, by= 1000)

# Bayesian Optimization
PARAM$correr_BO <- TRUE
# el diseno inicial son 12 evaluaciones (4 x 3 parametros): con 10 iteraciones se evalua solo el diseno inicial, sin evaluaciones guiadas por el modelo
PARAM$hyperparametertuning$iteraciones <- 10
# se usan solo si correr_BO es FALSE (vienen de una corrida anterior con otras variables y otros meses: reemplazar)
PARAM$hiperparametros_manuales <- list(
  num_iterations= 4028L, num_leaves= 1226L, min_sum_hessian_in_leaf= 0.00477222056195233
)

# ganancia por envio
PARAM$ganancia_acierto <- 1072500   # BAJA+2
PARAM$ganancia_error   <- -27500    # cualquier otra clase

In [5]:
# Parametros fijos del LightGBM; la BO pisa los tres de abajo
PARAM$lgbm$param_fijos <- list(
  boosting= "gbdt",
  objective= "binary",
  metric= "auc",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,          # fijo, muy importante
  feature_fraction= 0.50,       # fijo
  feature_fraction_bynode= 0.20,# fijo
  learning_rate= 0.005,         # fijo, muy importante

  num_iterations= 20000,     # se modificara
  num_leaves= 1024,          # se modificara
  min_sum_hessian_in_leaf= 1e-06  # se modificara
)

# Espacio de busqueda de la BO
PARAM$hyperparametertuning$hs <- ps(
  num_iterations= p_int(lower= 64L, upper= 4096L),
  num_leaves= p_int(lower= 16L, upper= 2048L),
  min_sum_hessian_in_leaf= p_dbl(lower= 1e-06, upper= 0.1)
)

## 3. Funciones auxiliares

In [6]:
# escribe una linea en el log de la BO (con encabezado la primera vez)
loguear <- function(reg, arch, verbose= TRUE) {
  linea <- paste0(format(Sys.time(), "%Y%m%d.%H%M%S"), "\t",
                  paste(unlist(reg), collapse= "\t"), "\n")
  if (!file.exists(arch)) {
    cat(paste0("fecha\t", paste(names(reg), collapse= "\t"), "\n"), file= arch)
  }
  cat(linea, file= arch, append= TRUE)
  if (verbose) cat(linea)
}

# ganancia acumulada para una cantidad de envios, ordenando por probabilidad descendente
curva_ganancia <- function(prob, clase_ternaria, cortes) {
  gan <- ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia_acierto, PARAM$ganancia_error)
  acumulada <- cumsum(gan[order(prob, decreasing= TRUE)])
  data.table(envios= cortes, ganancia= acumulada[cortes])
}

# cantidad de clientes y de bajas (BAJA+1 y BAJA+2) por mes, para comparar con el corte elegido
tabla_bajas <- function(meses) {
  dataset[foto_mes %in% meses, list(
    clientes = .N,
    baja1    = sum(clase_ternaria == "BAJA+1", na.rm= TRUE),
    baja2    = sum(clase_ternaria == "BAJA+2", na.rm= TRUE),
    bajas    = sum(clase_ternaria %in% c("BAJA+1", "BAJA+2"))
  ), by= foto_mes][order(foto_mes)]
}

## 4. Carga de datos y chequeos

Si el CSV ya trae `clase_ternaria` se usa tal cual; si no, se genera con la misma lógica (con todos los meses del archivo, antes de filtrar).

In [7]:
dataset <- fread(PARAM$archivo_dataset, stringsAsFactors= TRUE)
cat("filas:", nrow(dataset), "  columnas:", ncol(dataset), "\n")

filas: 983061   columnas: 154 


In [8]:
if (!("clase_ternaria" %in% names(dataset))) {
  cat("El dataset no tiene clase_ternaria: la genero\n")

  dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 + foto_mes%%100)]

  setorder(dsimple, numero_de_cliente, periodo0)

  periodo_ultimo <- dsimple[, max(periodo0)]
  periodo_anteultimo <- periodo_ultimo - 1

  dsimple[, c("periodo1", "periodo2") :=
            shift(periodo0, n= 1:2, fill= NA, type= "lead"), numero_de_cliente]

  dsimple[periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA"]

  dsimple[periodo0 < periodo_ultimo &
            (is.na(periodo1) | periodo0 + 1 < periodo1),
          clase_ternaria := "BAJA+1"]

  dsimple[periodo0 < periodo_anteultimo & (periodo0 + 1 == periodo1) &
            (is.na(periodo2) | periodo0 + 2 < periodo2),
          clase_ternaria := "BAJA+2"]

  setorder(dsimple, pos)
  dataset[, clase_ternaria := dsimple$clase_ternaria]
}

dataset[foto_mes >= 202103, .N, list(foto_mes, clase_ternaria)][order(foto_mes, clase_ternaria)]

El dataset no tiene clase_ternaria: la genero


foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


In [9]:
# chequeos: los meses existen, los de entrenamiento tienen clase, y agosto tiene un registro por cliente
meses_necesarios <- unique(c(PARAM$meses_fe, PARAM$train_bo, PARAM$validate_bo, PARAM$train_final, PARAM$future))
stopifnot(all(meses_necesarios %in% unique(dataset$foto_mes)))

stopifnot(!any(is.na(dataset[foto_mes %in% PARAM$train_final, clase_ternaria])))
stopifnot(!anyDuplicated(dataset[foto_mes %in% PARAM$future, numero_de_cliente]))

cat("clientes en el mes a predecir:", dataset[foto_mes %in% PARAM$future, .N], "\n")

clientes en el mes a predecir: 164647 


In [10]:
# clase binaria: BAJA+1 y BAJA+2 = 1, CONTINUA = 0 (queda NA donde no hay clase, por ejemplo agosto)
dataset[, clase01 := NA_integer_]
dataset[!is.na(clase_ternaria),
        clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

## 5. Feature engineering

Se hace **antes de cualquier entrenamiento**, en este orden:

1. **`ccajas_depositos` roto en 202105.** Todos sus valores son 0 ese mes, así que se pasa a `NA` solo en 202105. Se hace primero, para que los lags y deltas que dependen de ese valor también queden en `NA`.
2. **Lags y deltas 1 y 2** (marzo a agosto) para todas las variables, menos `foto_mes`, `numero_de_cliente` y la clase ternaria (y `clase01`). Esto incluye `cprestamos_personales` y `mprestamos_personales` (`PARAM$variables_fuera_bo`), que después se dejan afuera solo de la BO.
   * Los lags se calculan por **mes calendario** (el lag 1 de un registro es el mismo cliente un mes antes, el lag 2 dos meses antes). Si no hay registro de ese mes, queda `NA`: marzo no tiene lags y abril no tiene lag 2.
   * El delta es la **diferencia simple**: `delta1 = valor_t - valor_t-1` y `delta2 = valor_t - valor_t-2`.
3. **Ranking con cero fijo, dentro de cada `foto_mes`,** sobre las variables originales, los lags y los deltas: los valores negativos se ordenan entre sí y quedan en [-1, 0), los positivos en (0, 1], y el 0 queda en 0. Los `NA` se mantienen (no se imputa nada).
4. Se eliminan todas las variables sin rankear, **salvo `numero_de_cliente`, `foto_mes` y las clases**, que se conservan sin transformar: el modelo usa las nuevas (`_rank`, `_lag1_rank`, `_lag2_rank`, `_delta1_rank`, `_delta2_rank`) más `numero_de_cliente` y `foto_mes`.
5. Se guarda el dataset completo con el feature engineering (`PARAM$archivo_dataset_fe`).

In [11]:
# me quedo con los meses del feature engineering
dataset <- dataset[foto_mes %in% PARAM$meses_fe]

# paso 1: ccajas_depositos esta roto en 202105 (todos ceros) -> NA solo en ese mes
stopifnot("ccajas_depositos" %in% names(dataset))
dataset[foto_mes == 202105, ccajas_depositos := NA]

# paso 2: claves para encontrar el mismo cliente 1 y 2 meses antes (por mes calendario)
periodo <- (dataset$foto_mes %/% 100L) * 12L + dataset$foto_mes %% 100L
clave <- as.numeric(dataset$numero_de_cliente) * 1000 + (periodo - min(periodo))
stopifnot(!anyDuplicated(clave))

idx_lag1 <- match(clave - 1, clave)   # NA si no hay registro del mes anterior
idx_lag2 <- match(clave - 2, clave)   # NA si no hay registro de hace dos meses

# variables sobre las que se hace el feature engineering
cols_no_fe <- c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01")
vars_fe <- setdiff(names(dataset), cols_no_fe)
stopifnot(all(PARAM$variables_fuera_bo %in% vars_fe))   # los prestamos tambien pasan por el FE
stopifnot(all(sapply(vars_fe, function(v) is.numeric(dataset[[v]]))))
cat("variables para el feature engineering:", length(vars_fe), "\n")

variables para el feature engineering: 152 


In [ ]:
# ranking 

rank_cero_fijo <- function(x) {
  out <- rep(NA_real_, length(x))
  ok <- !is.na(x)
  out[ok & x == 0] <- 0

  pos <- which(ok & x > 0)
  if (length(pos) > 0) out[pos] <- frank(x[pos], ties.method= "average") / length(pos)

  neg <- which(ok & x < 0)
  if (length(neg) > 0) out[neg] <- -frank(-x[neg], ties.method= "average") / length(neg)

  out
}

# aplica el ranking dentro de cada foto_mes
filas_por_mes <- split(seq_len(nrow(dataset)), dataset$foto_mes)

rank_por_mes <- function(x) {
  out <- rep(NA_real_, length(x))
  for (ii in filas_por_mes) out[ii] <- rank_cero_fijo(x[ii])
  out
}

sufijos <- c("_rank", "_lag1_rank", "_lag2_rank", "_delta1_rank", "_delta2_rank")
nuevas <- vector("list", length(vars_fe) * length(sufijos))
nombres_nuevas <- character(length(nuevas))
k <- 0L

for (v in vars_fe) {
  x  <- as.numeric(dataset[[v]])
  l1 <- x[idx_lag1]
  l2 <- x[idx_lag2]
  versiones <- list(x, l1, l2, x - l1, x - l2)

  for (j in seq_along(versiones)) {
    k <- k + 1L
    nuevas[[k]] <- rank_por_mes(versiones[[j]])
    nombres_nuevas[k] <- paste0(v, sufijos[j])
  }
}
names(nuevas) <- nombres_nuevas

# elimino todas las variables sin rankear y agrego las rankeadas
cols_borrar <- setdiff(names(dataset), cols_no_fe)
dataset[, (cols_borrar) := NULL]
dataset[, (nombres_nuevas) := nuevas]

rm(nuevas, versiones, x, l1, l2, idx_lag1, idx_lag2, clave, periodo)
gc(full= TRUE, verbose= FALSE)

cat("filas:", nrow(dataset), "  columnas:", ncol(dataset), "\n")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2306419,123.2,4562020,243.7,3020678,161.4
Vcells,759798170,5796.9,1524645884,11632.2,1514343788,11553.6


filas: 983061   columnas: 764 


In [13]:
# chequeos del feature engineering
rangos <- suppressWarnings(dataset[, lapply(.SD, function(z) c(min(z, na.rm= TRUE), max(z, na.rm= TRUE))),
                                   .SDcols= nombres_nuevas])
rangos <- unlist(rangos)
rangos <- rangos[is.finite(rangos)]
stopifnot(all(rangos >= -1 & rangos <= 1))

# marzo no tiene lags; abril no tiene lag 2
cols_lag1 <- grep("_lag1_rank$", nombres_nuevas, value= TRUE)
cols_lag2 <- grep("_lag2_rank$", nombres_nuevas, value= TRUE)
stopifnot(all(sapply(cols_lag1, function(v) dataset[foto_mes == 202103, all(is.na(get(v)))])))
stopifnot(all(sapply(cols_lag2, function(v) dataset[foto_mes %in% c(202103, 202104), all(is.na(get(v)))])))

# ccajas_depositos: NA en 202105, y por arrastre en el lag 1 de junio, el lag 2 de julio y los deltas que dependen de mayo
stopifnot(dataset[foto_mes == 202105, all(is.na(ccajas_depositos_rank))])
stopifnot(dataset[foto_mes == 202106, all(is.na(ccajas_depositos_lag1_rank))])
stopifnot(dataset[foto_mes == 202107, all(is.na(ccajas_depositos_lag2_rank))])

# numero_de_cliente y foto_mes: siguen en el dataset, sin ranking ni lags (valores originales)
stopifnot(all(c("numero_de_cliente", "foto_mes") %in% names(dataset)))
stopifnot(!any(grepl("^(numero_de_cliente|foto_mes)_", names(dataset))))
stopifnot(all(dataset$foto_mes %in% PARAM$meses_fe), !anyNA(dataset$numero_de_cliente))

# los prestamos pasaron por el FE (5 columnas cada uno)
cols_prestamos <- as.vector(outer(PARAM$variables_fuera_bo, sufijos, paste0))
stopifnot(all(cols_prestamos %in% names(dataset)))

cat("variables nuevas:", length(nombres_nuevas), "\n")
cat("proporcion de NA por mes (promedio de todas las variables nuevas):\n")
print(dataset[, list(prop_NA= mean(is.na(unlist(.SD)))), by= foto_mes, .SDcols= nombres_nuevas][order(foto_mes)])

variables nuevas: 760 
proporcion de NA por mes (promedio de todas las variables nuevas):
   foto_mes    prop_NA
      <int>      <num>
1:   202103 0.81375918
2:   202104 0.44471777
3:   202105 0.08294619
4:   202106 0.08140961
5:   202107 0.07905588
6:   202108 0.07660612


### Guardado del dataset con feature engineering

Se guarda el dataset **completo** (`numero_de_cliente`, `foto_mes`, `clase_ternaria`, `clase01` y todas las variables rankeadas, incluidos los préstamos) en la misma carpeta que el archivo original. Para el entrenamiento final se puede volver a cargar con `fread(PARAM$archivo_dataset_fe, stringsAsFactors= TRUE)` sin repetir el feature engineering.

In [14]:
fwrite(dataset, PARAM$archivo_dataset_fe)
cat("dataset con FE guardado en:", PARAM$archivo_dataset_fe, "\n")
cat("filas:", nrow(dataset), "  columnas:", ncol(dataset), "\n")

dataset con FE guardado en: /home/ds/buckets/b1/datasets/competencia_01_crudo_FE_rank.csv 
filas: 983061   columnas: 764 


## 6. Diagnóstico de Data Drift

Se compara la distribución de cada variable rankeada entre **junio y agosto** con el estadístico de Kolmogorov-Smirnov (0 = iguales, 1 = totalmente distintas). Como referencia se incluye también **abril vs junio**, que muestra cuánto cambia una variable "normalmente" en dos meses.

La tabla es informativa: **no excluye nada por sí sola**. Si alguna variable muestra un salto muy grande en `ks_futuro` respecto de `ks_referencia`, agregar su nombre exacto (por ejemplo `"mcuentas_saldo_lag1_rank"`) a `PARAM$columnas_excluidas_modelo` y volver a correr desde la celda de parámetros. Los `NA` se ignoran en cada comparación.

In [ ]:
set.seed(PARAM$semilla_primigenia)

muestra_mes <- function(mes, n= 20000L) {
  x <- dataset[foto_mes == mes]
  x[sample(.N, min(.N, n))]
}

d_abr <- muestra_mes(202104)
d_jun <- muestra_mes(202106)
d_ago <- muestra_mes(PARAM$future[1])

vars_num <- grep("_rank$", names(dataset), value= TRUE)

ks_stat <- function(a, b) {
  tryCatch(
    suppressWarnings(as.numeric(ks.test(a, b, exact= FALSE)$statistic)),
    error= function(e) NA_real_
  )
}

drift <- rbindlist(lapply(vars_num, function(v) {
  data.table(
    variable = v,
    ks_referencia = ks_stat(d_abr[[v]], d_jun[[v]]),   # abril vs junio
    ks_futuro     = ks_stat(d_jun[[v]], d_ago[[v]])    # junio vs agosto
  )
}))

setorder(drift, -ks_futuro)
print(head(drift, 25))

rm(d_abr, d_jun, d_ago)

                             variable ks_referencia ks_futuro
                               <char>         <num>     <num>
 1:        ccajas_depositos_lag1_rank            NA        NA
 2:      ccajas_depositos_delta1_rank            NA        NA
 3:    cliente_antiguedad_delta1_rank     1.0000000 1.0000000
 4:        Visa_fechaalta_delta2_rank            NA 1.0000000
 5:    cliente_antiguedad_delta2_rank            NA 1.0000000
 6:      Master_fechaalta_delta2_rank            NA 0.9999432
 7:      Master_fechaalta_delta1_rank     1.0000000 0.9998877
 8:        Visa_fechaalta_delta1_rank     0.9999469 0.9998407
 9: Master_fultimo_cierre_delta1_rank     0.9995500 0.9994942
10:   Visa_fultimo_cierre_delta1_rank     0.9995218 0.9994687
11: Master_fultimo_cierre_delta2_rank            NA 0.9993195
12:   Visa_fultimo_cierre_delta2_rank            NA 0.9991429
13:            ccuenta_corriente_rank     0.9977500 0.9973500
14:       ccuenta_corriente_lag1_rank     0.9977311 0.9973270
15:     

## 7. Preprocesamiento

Se arman los datasets de la Bayesian Optimization, **sin undersampling**:
* **Entrenamiento:** marzo + abril completos.
* **Validación:** junio completo.

Hay dos conjuntos de variables:
* `campos_bo`: las que usa la BO (`numero_de_cliente`, `foto_mes` y las rankeadas, **sin** los préstamos).
* `campos_modelo`: las que usan la curva de junio y el modelo final (`campos_bo` **más** los préstamos).

In [16]:
# variables del modelo (curva de junio y modelo final): numero_de_cliente, foto_mes (sin FE) y todas las rankeadas, prestamos incluidos
campos_modelo <- c(
  "numero_de_cliente", "foto_mes",
  setdiff(grep("_rank$", colnames(dataset), value= TRUE), PARAM$columnas_excluidas_modelo)
)

# variables de la BO: las mismas, sin los prestamos
cols_prestamos <- as.vector(outer(PARAM$variables_fuera_bo, sufijos, paste0))
campos_bo <- setdiff(campos_modelo, cols_prestamos)

stopifnot(all(c("numero_de_cliente", "foto_mes") %in% campos_bo))
stopifnot(!any(cols_prestamos %in% campos_bo))
cat("variables de la BO:", length(campos_bo), "\n")
cat("variables de la curva de junio y del modelo final:", length(campos_modelo), "\n")

variables de la BO: 752 
variables de la curva de junio y del modelo final: 762 


In [17]:
# training de la BO: marzo + abril, todos los clientes
dtr  <- dataset[foto_mes %in% PARAM$train_bo]

# validacion de la BO: junio completo
dval <- dataset[foto_mes %in% PARAM$validate_bo]

# datasets de la BO: sin los prestamos
dtrain_bo <- lgb.Dataset(
  data= data.matrix(dtr[, campos_bo, with= FALSE]),
  label= dtr$clase01,
  free_raw_data= FALSE
)

dvalidate_bo <- lgb.Dataset(
  data= data.matrix(dval[, campos_bo, with= FALSE]),
  label= dval$clase01,
  free_raw_data= FALSE
)

# dataset de entrenamiento de la curva de junio: con los prestamos
dtrain <- lgb.Dataset(
  data= data.matrix(dtr[, campos_modelo, with= FALSE]),
  label= dtr$clase01,
  free_raw_data= FALSE
)

cat("train BO:", nrow(dtr), "filas,", sum(dtr$clase01), "bajas\n")
cat("validacion BO:", nrow(dval), "filas,", sum(dval$clase01), "bajas\n")

cat("\nbajas (BAJA+1 y BAJA+2) por mes:\n")
print(tabla_bajas(c(202103, 202104, 202105, 202106)))

train BO: 326184 filas, 4082 bajas
validacion BO: 164114 filas, 1972 bajas

bajas (BAJA+1 y BAJA+2) por mes:
   foto_mes clientes baja1 baja2 bajas
      <int>    <int> <int> <int> <int>
1:   202103   162900  1019   960  1979
2:   202104   163284   964  1139  2103
3:   202105   163768  1143   870  2013
4:   202106   164114   874  1098  1972


## 8. Optimización Bayesiana

Una sola semilla (`PARAM$semilla_primigenia`) y métrica **AUC** sobre junio.

In [18]:
# carpeta del experimento
carpeta_exp <- file.path(PARAM$carpeta_base, "experimentos", PARAM$experimento)
dir.create(carpeta_exp, recursive= TRUE, showWarnings= FALSE)
setwd(carpeta_exp)

archivo_log <- "BO_log.txt"

# si hay un log de una corrida anterior lo renombro (no se borra) para no mezclar corridas
if (PARAM$correr_BO && file.exists(archivo_log)) {
  file.rename(archivo_log, paste0(archivo_log, ".", format(Sys.time(), "%Y%m%d%H%M%S"), ".bak"))
}

giter <- 0

In [19]:
# En xs llegan los hiperparametros que propone la BO
Estimar_lightgbm <- function(xs) {

  flush.console()
  giter <<- giter + 1

  # xs pisa a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, xs)

  modelo <- lgb.train(
    data= dtrain_bo,
    valids= list("val" = dvalidate_bo),
    param= param_completo
  )

  xx <- param_completo
  xx$iter <- giter
  xx$metrica <- modelo$best_score

  loguear(xx, archivo_log)
  set.seed(PARAM$semilla_primigenia, kind= "L'Ecuyer-CMRG")  # reordeno la semilla para mlr3mbo

  return(list(metrica = xx$metrica))
}

In [20]:
objective <- ObjectiveRFun$new(
  fun= Estimar_lightgbm,
  domain= PARAM$hyperparametertuning$hs,
  codomain= ps(metrica = p_dbl(tags= "maximize")),
  id= "BO_lightgbm"
)

instance <- OptimInstanceBatchSingleCrit$new(
  objective= objective,
  terminator= trm("evals", n_evals= PARAM$hyperparametertuning$iteraciones)
)

surrogate <- srlrn(lrn("regr.km",
  optim.method= "BFGS",
  nugget.estim= TRUE,
  control= list(trace= FALSE)))

acq_function <- acqf("ei")   # expected improvement

acq_optimizer <- acqo(
  optimizer= opt("random_search", batch_size= 1000),
  terminator= trm("evals", n_evals= 1000)
)

optimizer <- opt("mbo",
  loop_function= bayesopt_ego,
  surrogate= surrogate,
  acq_function= acq_function,
  acq_optimizer= acq_optimizer
)

### Corrida

Con `PARAM$correr_BO <- TRUE` esta celda es la más lenta del notebook. Si se pone en `FALSE`, se saltea y se usan `PARAM$hiperparametros_manuales`.

In [21]:
if (PARAM$correr_BO) {
  bayesiana_salida <- optimizer$optimize(instance)

  tb_bayesiana <- fread(archivo_log)
  setorder(tb_bayesiana, -metrica)
  print(head(tb_bayesiana[, list(iter, num_iterations, num_leaves, min_sum_hessian_in_leaf, metrica)], 10))

  mejores_hiperparametros <- as.list(
    tb_bayesiana[1, list(num_iterations, num_leaves, min_sum_hessian_in_leaf)]
  )
} else {
  mejores_hiperparametros <- PARAM$hiperparametros_manuales
}

mejores_hiperparametros

$num_iterations
[1] 4028

$num_leaves
[1] 1226

$min_sum_hessian_in_leaf
[1] 0.004772221

## 9. Curva de ganancia de junio

Agosto no tiene clase, así que no se puede medir su ganancia. Lo que sí se puede es repetir el ejercicio en junio:
1. Se entrenan `PARAM$n_semillas` modelos (una semilla cada uno) con los mejores hiperparámetros, usando **marzo + abril** y las variables de `campos_modelo` (con los préstamos).
2. Se promedian las probabilidades de junio y se calcula la ganancia para cada cantidad de envíos entre 8.000 y 15.000.
3. Se suaviza la curva con una media móvil de 5 puntos (±500 envíos) para no quedarse con un pico producto del azar, y se toma el máximo.

Esta curva es solo para elegir el corte, no para la optimización.

In [ ]:
# ================== CURVA DE JUNIO: ENTRENAMIENTO + GRAFICO ==================
param_final <- modifyList(PARAM$lgbm$param_fijos, mejores_hiperparametros)

X_val <- data.matrix(dval[, campos_modelo, with= FALSE])

semillas_curva <- PARAM$semillas 
prob_semillas  <- matrix(0, nrow= nrow(dval), ncol= length(semillas_curva))

# ---- entrenamiento ----
for (i in seq_along(semillas_curva)) {
  t0 <- Sys.time()
  param_semilla <- modifyList(param_final,
                              list(seed= semillas_curva[i],
                                   num_threads= 4))
  modelo_val <- lgb.train(data= dtrain, param= param_semilla)
  prob_semillas[, i] <- predict(modelo_val, X_val)
  cat("curva de junio: semilla", i, "de", length(semillas_curva),
      "- tardó", format(Sys.time() - t0), "\n")
  flush.console()
  saveRDS(prob_semillas, "prob_semillas_parcial.rds")
}
prob_val <- rowMeans(prob_semillas)   # ensamble

# ---- parametros de la curva ----
paso        <- 50
cortes_plot <- seq(3500, 20500, by= paso)
ventana     <- 21                    # +-500 envios
tolerancia  <- 0.01                  # meseta: >= (1 - tolerancia) * maximo

suavizar <- function(g) frollmean(g, n= ventana, align= "center")

# ---- curvas ----
g_ens <- suavizar(curva_ganancia(prob_val, dval$clase_ternaria, cortes_plot)$ganancia)

g_sueltos <- rowMeans(sapply(seq_len(ncol(prob_semillas)), function(i) {
  suavizar(curva_ganancia(prob_semillas[, i], dval$clase_ternaria, cortes_plot)$ganancia)
}))

# ---- maximo suavizado y meseta contigua ----
k_junio <- cortes_plot[which.max(g_ens)]

umbral <- (1 - tolerancia) * max(g_ens, na.rm= TRUE)
i_max  <- which.max(g_ens)
ini <- i_max
while (ini > 1 && !is.na(g_ens[ini - 1]) && g_ens[ini - 1] >= umbral) ini <- ini - 1
fin <- i_max
while (fin < length(g_ens) && !is.na(g_ens[fin + 1]) && g_ens[fin + 1] >= umbral) fin <- fin + 1
meseta_ini    <- cortes_plot[ini]
meseta_fin    <- cortes_plot[fin]
centro_meseta <- round((meseta_ini + meseta_fin) / 2)

# ---- grafico ----
ylim <- range(c(g_ens, g_sueltos), na.rm= TRUE) / 1e6

plot(cortes_plot, g_ens / 1e6, type= "l", lwd= 3, xlim= c(4000, 20000), ylim= ylim,
     xlab= "envíos en junio", ylab= "ganancia suavizada (M)",
     main= "Curva de ganancia del ensamble en junio")
lines(cortes_plot, g_sueltos / 1e6, lty= 2, lwd= 1.5)
abline(v= c(meseta_ini, meseta_fin), col= "gray60", lty= 3)
abline(v= centro_meseta, col= "red", lty= 2, lwd= 2)
abline(v= k_junio, col= "blue", lty= 1, lwd= 1.5)

legend("bottomright", bty= "n",
       legend= c("ensamble", "promedio de sueltos", "centro de la meseta", "máximo suavizado"),
       lty= c(1, 2, 2, 1), lwd= c(3, 1.5, 2, 1.5), col= c("black", "black", "red", "blue"))

cat("meseta entre", meseta_ini, "y", meseta_fin, "envíos\n")
cat("centro de la meseta:", centro_meseta, "envíos\n")
cat("corte de junio (máximo suavizado):", k_junio, "envíos\n")

## 10. Modelo final y predicción de agosto

Se entrena con **marzo + abril + mayo + junio**, sin undersampling y con los mismos hiperparámetros, un modelo por semilla (`PARAM$n_semillas`). Cada uno predice agosto y la **probabilidad final es el promedio** de las probabilidades de todos los modelos.

In [23]:
dfinal <- dataset[foto_mes %in% PARAM$train_final]
cat("train final:", nrow(dfinal), "filas,", sum(dfinal$clase01), "bajas\n")

dtrain_final <- lgb.Dataset(
  data= data.matrix(dfinal[, campos_modelo, with= FALSE]),
  label= dfinal$clase01,
  free_raw_data= FALSE
)

dfuture <- dataset[foto_mes %in% PARAM$future]
X_fut <- data.matrix(dfuture[, campos_modelo, with= FALSE])

# Carpeta para las probabilidades de cada semilla
dir_probs <- "~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas/probs_semillas"
dir.create(dir_probs, showWarnings= FALSE, recursive= TRUE)

prob_semillas <- matrix(
  NA_real_, nrow= nrow(dfuture), ncol= length(PARAM$semillas),
  dimnames= list(NULL, paste0("semilla_", PARAM$semillas))
)

for (i in seq_along(PARAM$semillas)) {
  param_semilla <- modifyList(param_final, list(seed= PARAM$semillas[i]))
  modelo_final <- lgb.train(data= dtrain_final, param= param_semilla)
  prob_semillas[, i] <- predict(modelo_final, X_fut)

  # Guardo las probabilidades de esta semilla
  fwrite(
    data.table(numero_de_cliente= dfuture$numero_de_cliente,
               prob= prob_semillas[, i]),
    file.path(dir_probs, paste0("prob_semilla_", PARAM$semillas[i], ".csv"))
  )

  cat("modelo final: semilla", i, "de", length(PARAM$semillas), "\n")
}

train final: 654066 filas, 8067 bajas
modelo final: semilla 1 de 10 
modelo final: semilla 2 de 10 
modelo final: semilla 3 de 10 
modelo final: semilla 4 de 10 
modelo final: semilla 5 de 10 
modelo final: semilla 6 de 10 
modelo final: semilla 7 de 10 
modelo final: semilla 8 de 10 
modelo final: semilla 9 de 10 
modelo final: semilla 10 de 10 


In [24]:
dir_base  <- "~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas"
dir_probs <- file.path(dir_base, "probs_semillas")

# Leo TODOS los archivos de probabilidades por semilla
archivos <- list.files(dir_probs, pattern = "^prob_semilla_.*\\.csv$", full.names = TRUE)
cat("Archivos encontrados:", length(archivos), "\n")
stopifnot(length(archivos) > 0)

lista <- lapply(archivos, function(f) {
  x <- fread(f)
  setnames(x, "prob", paste0("prob_", tools::file_path_sans_ext(basename(f))))
  setkey(x, numero_de_cliente)
  x
})

# Chequeo que todos tengan los mismos clientes
n_clientes <- sapply(lista, nrow)
stopifnot(length(unique(n_clientes)) == 1)

# Uno por numero_de_cliente y promedio simple
tb_futuro <- Reduce(function(a, b) merge(a, b, by = "numero_de_cliente"), lista)
stopifnot(nrow(tb_futuro) == n_clientes[1])

cols_prob <- grep("^prob_prob_semilla_", names(tb_futuro), value = TRUE)
tb_futuro[, prob := rowMeans(.SD), .SDcols = cols_prob]
cat("Semillas promediadas:", length(cols_prob), "\n")

# Ordeno por probabilidad descendente y guardo el ranking
setorder(tb_futuro, -prob)
fwrite(tb_futuro[, .(numero_de_cliente, prob)],
       file.path(dir_base, "ranking_prob_promedio.csv"))

cat("Ranking guardado:", nrow(tb_futuro), "clientes\n")

Archivos encontrados: 20 
Semillas promediadas: 20 
Ranking guardado: 164647 clientes


In [25]:
dir_base <- "~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas"

# Puntos de corte manuales (cantidad de envíos)
cortes <- c(10000, 11000, 11500, 12000, 12040, 12500, 13000)

# Leo el ranking y me aseguro de que esté ordenado
tb_futuro <- fread(file.path(dir_base, "ranking_prob_promedio.csv"))
setorder(tb_futuro, -prob)

# Un archivo por corte
for (k in cortes) {
  entrega <- tb_futuro[, .(numero_de_cliente)]
  entrega[, Predicted := 0L]
  entrega[1:k, Predicted := 1L]

  archivo <- file.path(dir_base, paste0("entrega_", k, "_envios.csv"))
  fwrite(entrega, archivo)
  cat("Guardado:", archivo, "| positivos:", sum(entrega$Predicted),
      "| prob del último enviado:", round(tb_futuro$prob[k], 5), "\n")
}

Guardado: ~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas/entrega_10000_envios.csv | positivos: 10000 | prob del último enviado: 0.00024 
Guardado: ~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas/entrega_11000_envios.csv | positivos: 11000 | prob del último enviado: 0.00019 
Guardado: ~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas/entrega_11500_envios.csv | positivos: 11500 | prob del último enviado: 0.00017 
Guardado: ~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas/entrega_12000_envios.csv | positivos: 12000 | prob del último enviado: 0.00016 
Guardado: ~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas/entrega_12040_envios.csv | positivos: 12040 | prob del último enviado: 0.00016 
Guardado: ~/buckets/b1/experimentos/primer_entrega_piloto_FE_rank_20semillas/entrega_12500_envios.csv | positivos: 12500 | prob del último enviado: 0.00015 
Guardado: ~/buckets/b1/experimentos/primer_entrega_piloto_

## 11. Archivos de envío

* **CSV por defecto** (`<experimento>.csv`): el corte de junio (curva suavizada) se escala al tamaño de agosto y se limita a [8.000, 15.000] (o se usa `PARAM$envios_manual` si se definió).
* **Un CSV por cada corte** de `PARAM$cortes_envios` (`<experimento>_k<corte>.csv`) para poder probar distintos puntos de corte.

Todos son **solo la columna de ids, sin encabezado**, ordenados por la probabilidad promedio. Los ids se pasan a **entero** y se escriben con `sprintf("%d")`, que no usa notación científica.

In [ ]:
dir_base  <- "/Users/sofiaforni/Documents/FCEN UBA Maestria/Economia y finanazas/prediccion_primera_competencia"
dir_probs <- file.path(dir_base, "probabilidades_primera_entrega")

archivos <- list.files(dir_probs, pattern = "prob_semilla_\\d+\\.csv$", full.names = TRUE)
cat("Archivos encontrados:", length(archivos), "\n")
stopifnot(length(archivos) > 0)

lista <- lapply(archivos, function(f) {
  semilla <- sub(".*prob_semilla_(\\d+)\\.csv$", "\\1", basename(f))
  x <- fread(f)
  setnames(x, "prob", paste0("p_", semilla))
  setkey(x, numero_de_cliente)
  x
})

# Mismos clientes en todos los archivos
ids_ref <- lista[[1]]$numero_de_cliente
stopifnot(all(sapply(lista, function(x) identical(x$numero_de_cliente, ids_ref))))

tb_futuro <- Reduce(function(a, b) merge(a, b, by = "numero_de_cliente"), lista)

cols_prob <- grep("^p_\\d+$", names(tb_futuro), value = TRUE)
tb_futuro[, prob := rowMeans(.SD), .SDcols = cols_prob]
cat("Semillas promediadas:", length(cols_prob), "\n")

setorder(tb_futuro, -prob)
fwrite(tb_futuro[, .(numero_de_cliente, prob)],
       file.path(dir_base, "ranking_prob_promedio.csv"))
cat("Ranking guardado:", nrow(tb_futuro), "clientes\n")

In [ ]:
library(data.table)

dir_base <- "/Users/sofiaforni/Documents/FCEN UBA Maestria/Economia y finanazas/prediccion_primera_competencia"

dir_entregas <- file.path(dir_base, "entregas")
dir.create(dir_entregas, showWarnings = FALSE)

cortes <- c(10000, 11000, 11500, 12000, 12040, 12700, 12350)

tb_futuro <- fread(file.path(dir_base, "ranking_prob_promedio.csv"))
setorder(tb_futuro, -prob)
stopifnot(max(cortes) <= nrow(tb_futuro))

for (k in cortes) {
  archivo <- file.path(dir_entregas, paste0("entrega_", k, "_envios.csv"))
  fwrite(tb_futuro[1:k, .(numero_de_cliente)], archivo, col.names = FALSE)
  cat("Guardado:", basename(archivo), "| ids enviados:", k,
      "| prob del último enviado:", round(tb_futuro$prob[k], 5), "\n")
}